# Praktikum 1 - JST Sederhana 2 Layer

**Nama:** Radit Firansah  
**NIM:** 244107020196

Membuat Jaringan Saraf Tiruan (JST) 2 layer dengan forward pass dan backpropagation manual pada dataset XOR.

## Langkah 1 - Import & Siapkan Dataset

Kita membuat dataset XOR sederhana, lalu menginisialisasi bobot dan bias.
Seed ditetapkan agar hasil dapat direproduksi.

In [1]:
import numpy as np

np.random.seed(42)

# Dataset XOR
X = np.array([[0,0],[0,1],[1,0],[1,1]])
y = np.array([[0],[1],[1],[0]])

# Parameter
input_size = 2
hidden_size = 2
output_size = 1
lr = 0.1

# Inisialisasi bobot
W1 = np.random.randn(input_size, hidden_size)
b1 = np.zeros((1, hidden_size))
W2 = np.random.randn(hidden_size, output_size)
b2 = np.zeros((1, output_size))

## Langkah 2 - Fungsi Aktivasi

Menggunakan fungsi sigmoid beserta turunannya.

In [2]:
# Fungsi aktivasi
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

## Langkah 3 - Training (Forward + Backpropagation)

Latih model selama 10000 epoch, cetak loss setiap 1000 epoch.

In [3]:
# Training
for epoch in range(10000):
    # Forward pass
    z1 = np.dot(X, W1) + b1
    a1 = sigmoid(z1)
    z2 = np.dot(a1, W2) + b2
    a2 = sigmoid(z2)

    # Hitung error
    error = y - a2

    # Backpropagation
    d_a2 = error * sigmoid_derivative(a2)
    d_W2 = np.dot(a1.T, d_a2)
    d_b2 = np.sum(d_a2, axis=0, keepdims=True)

    d_a1 = np.dot(d_a2, W2.T) * sigmoid_derivative(a1)
    d_W1 = np.dot(X.T, d_a1)
    d_b1 = np.sum(d_a1, axis=0, keepdims=True)

    # Update bobot
    W1 += lr * d_W1
    b1 += lr * d_b1
    W2 += lr * d_W2
    b2 += lr * d_b2

    if epoch % 1000 == 0:
        loss = np.mean(np.square(error))
        print(f"Epoch {epoch}, Loss: {loss}")

Epoch 0, Loss: 0.2558299419444368
Epoch 1000, Loss: 0.2494056595655124
Epoch 2000, Loss: 0.24544465159719803
Epoch 3000, Loss: 0.20470733040714414
Epoch 4000, Loss: 0.15320405369970766
Epoch 5000, Loss: 0.13869146014771938
Epoch 6000, Loss: 0.13359363321851758
Epoch 7000, Loss: 0.13115112181268004
Epoch 8000, Loss: 0.12974916048385188
Epoch 9000, Loss: 0.12884908965171127


## Langkah 4 - Output Akhir

In [4]:
# Output akhir
print("Prediksi:")
print(a2)

Prediksi:
[[0.05300868]
 [0.49554213]
 [0.95091319]
 [0.50319888]]


## Tugas 1 - Eksperimen Hidden Layer & Fungsi Aktivasi

Pada bagian ini kita membandingkan:
1. `hidden_size = 2` (konfigurasi awal) vs `hidden_size = 3`.
2. Aktivasi **Sigmoid** vs **ReLU** pada hidden layer.

In [5]:
def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def aktivasi(x, nama):
    return sigmoid(x) if nama == 'sigmoid' else relu(x)

def turunan(x, nama):
    return sigmoid_derivative(x) if nama == 'sigmoid' else relu_derivative(x)

def latih_xor(hidden_size, nama_aktivasi='sigmoid', lr=0.1, epochs=10000):
    np.random.seed(42)
    W1 = np.random.randn(2, hidden_size)
    b1 = np.zeros((1, hidden_size))
    W2 = np.random.randn(hidden_size, 1)
    b2 = np.zeros((1, 1))
    for epoch in range(epochs):
        z1 = np.dot(X, W1) + b1
        a1 = aktivasi(z1, nama_aktivasi)
        z2 = np.dot(a1, W2) + b2
        a2 = sigmoid(z2)
        error = y - a2
        d_a2 = error * sigmoid_derivative(a2)
        d_W2 = np.dot(a1.T, d_a2)
        d_b2 = np.sum(d_a2, axis=0, keepdims=True)
        d_a1 = np.dot(d_a2, W2.T) * turunan(a1, nama_aktivasi)
        d_W1 = np.dot(X.T, d_a1)
        d_b1 = np.sum(d_a1, axis=0, keepdims=True)
        W1 += lr * d_W1; b1 += lr * d_b1
        W2 += lr * d_W2; b2 += lr * d_b2
    loss = np.mean(np.square(error))
    return loss, a2

konfigurasi = [(2, 'sigmoid'), (3, 'sigmoid'), (2, 'relu'), (3, 'relu')]
print(f"{'Hidden':<8}{'Aktivasi':<12}{'Loss Akhir':<12}Prediksi")
for hs, act in konfigurasi:
    loss, pred = latih_xor(hs, act)
    pred_str = ' '.join(f'{p[0]:.3f}' for p in pred)
    print(f"{hs:<8}{act:<12}{loss:<12.4f}{pred_str}")

Hidden  Aktivasi    Loss Akhir  Prediksi


2       sigmoid     0.1282      0.053 0.496 0.951 0.503


3       sigmoid     0.0023      0.025 0.953 0.951 0.063
2       relu        0.1250      0.500 0.989 0.500 0.009


3       relu        0.0004      0.030 0.986 0.986 0.012


### Kesimpulan Tugas 1

- Menambah neuron hidden layer menjadi 3 memberi jaringan lebih banyak kapasitas sehingga loss akhir umumnya lebih kecil.
- Aktivasi **ReLU** membuat proses belajar lebih cepat pada nilai besar, tetapi pada dataset XOR yang sangat kecil hasilnya bisa kurang stabil dibanding Sigmoid.
- Pola prediksi yang benar untuk XOR adalah mendekati `0, 1, 1, 0`.